# Phase 0: Agent & Model Availability Check

Phase 1 실험 전에 각 agent CLI가 설치되어 있는지, 어떤 모델이 사용 가능한지 확인합니다.

## 목적

- 각 agent(codex, cursor, aider, antigravity, copilot)의 **CLI 설치 여부** 확인
- 사용 가능한 **모델 카탈로그** 조회 (지원하는 agent만)
- Phase 1에서 사용할 **agent × model × reasoning 조합 예시** 정리
- 업데이트 후 모델 변경/추가/삭제 여부를 빠르게 파악


## Cell 1 : 공통 설정 및 헬퍼


In [55]:
# cell 1 : 공통 설정 및 헬퍼
import json
import shutil
import subprocess
import sys
from datetime import datetime

CHECK_DATE = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
print(f"Phase 0 Check Date: {CHECK_DATE}")
print(f"Python: {sys.executable}")
print(f"Version: {sys.version}")


def _needs_shell(name: str) -> bool:
    """Windows .CMD/.BAT wrapper는 shell=True로 실행해야 한다."""
    path = shutil.which(name)
    if not path:
        return False
    return path.lower().endswith(('.cmd', '.bat'))


def check_cli(name: str) -> dict:
    """CLI 존재 여부 및 버전 확인."""
    path = shutil.which(name)
    result = {"cli": name, "installed": path is not None, "path": path, "version": None, "error": None}
    if not path:
        result["error"] = f"{name} not found in PATH"
        return result
    try:
        use_shell = _needs_shell(name)
        args = f"{name} --version" if use_shell else [name, "--version"]
        proc = subprocess.run(
            args, shell=use_shell,
            capture_output=True, text=True, encoding="utf-8", errors="replace", timeout=30,
        )
        result["version"] = (proc.stdout.strip() or proc.stderr.strip())[:200]
    except Exception as exc:
        result["error"] = str(exc)[:200]
    return result


def run_quiet(args, *, timeout=60):
    """명령 실행 후 (returncode, stdout, stderr) 반환. .CMD 래퍼 자동 처리."""
    try:
        cmd_name = args[0] if isinstance(args, (list, tuple)) else args.split()[0]
        use_shell = _needs_shell(cmd_name)
        if use_shell and isinstance(args, (list, tuple)):
            # list를 shell 문자열로 변환
            args = subprocess.list2cmdline(args)
        proc = subprocess.run(
            args, shell=use_shell, capture_output=True, text=True,
            encoding="utf-8", errors="replace", timeout=timeout,
        )
        return proc.returncode, proc.stdout, proc.stderr
    except Exception as exc:
        return -1, "", str(exc)


def print_status(name, installed, version=None, error=None):
    icon = "✅" if installed else "❌"
    print(f"  {icon} {name}: ", end="")
    if installed:
        print(version or "(version unknown)")
    else:
        print(error or "not found")


Phase 0 Check Date: 2026-04-25 09:55:13
Python: c:\git\docs_experiment\.venv\Scripts\python.exe
Version: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]


## Cell 2 : 전체 Agent CLI 존재 여부 확인


In [56]:
# cell 2 : 전체 Agent CLI 존재 여부 확인
AGENTS_TO_CHECK = [
    ("codex",        "npm i -g @openai/codex"),
    ("cursor-agent", "irm 'https://cursor.com/install?win32=true' | iex"),
    ("aider",        "pip install aider-chat"),
    ("antigravity",  "Antigravity IDE 설치 후 PATH에 추가"),
    ("gh",           "https://cli.github.com/ (Copilot용)"),
]

cli_results = {}
print("=" * 60)
print("Agent CLI 설치 현황")
print("=" * 60)
for cli_name, install_hint in AGENTS_TO_CHECK:
    r = check_cli(cli_name)
    cli_results[cli_name] = r
    print_status(cli_name, r["installed"], r["version"], r["error"])
    if not r["installed"]:
        print(f"       설치: {install_hint}")

print()
available_agents = [name for name, r in cli_results.items() if r["installed"]]
print(f"사용 가능 agent: {available_agents or '없음'}")


Agent CLI 설치 현황
  ✅ codex: codex-cli 0.122.0
  ❌ cursor-agent: cursor-agent not found in PATH
       설치: irm 'https://cursor.com/install?win32=true' | iex
  ❌ aider: aider not found in PATH
       설치: pip install aider-chat
  ✅ antigravity: 1.107.0
15487b3041e65228cae24980a3f796c905ef582c
x64
  ✅ gh: gh version 2.86.0 (2026-01-21)
https://github.com/cli/cli/releases/tag/v2.86.0

사용 가능 agent: ['codex', 'antigravity', 'gh']


## Cell 3 : Codex — 모델 카탈로그 조회


In [57]:
# cell 3 : Codex — 모델 카탈로그 조회
print("=" * 60)
print("Codex 모델 카탈로그")
print("=" * 60)

codex_models = []
if not cli_results.get("codex", {}).get("installed"):
    print("❌ codex CLI 미설치 — 건너뜀")
else:
    rc, stdout, stderr = run_quiet(["codex", "debug", "models"], timeout=60)
    if rc != 0:
        print(f"⚠️ codex debug models 실패 (exit={rc})")
        if stderr:
            print(stderr[:500])
    else:
        try:
            catalog = json.loads(stdout)
            models = catalog.get("models", [])
            for m in models:
                slug = m.get("slug", "?")
                vis = m.get("visibility", "?")
                reasoning = [x.get("effort") for x in (m.get("supported_reasoning_levels") or [])]
                default_r = m.get("default_reasoning_level", "-")
                codex_models.append({
                    "slug": slug, "visibility": vis,
                    "reasoning_levels": reasoning, "default_reasoning": default_r,
                })
                icon = "🟢" if vis == "list" else "⚪"
                print(f"  {icon} {slug}  (visibility={vis}, reasoning={reasoning or ['default']}, default={default_r})")
            print(f"\n총 {len(models)}개 모델")
        except json.JSONDecodeError:
            print("⚠️ JSON 파싱 실패")
            print(stdout[:500])


Codex 모델 카탈로그
  🟢 gpt-5.3-codex  (visibility=list, reasoning=['low', 'medium', 'high', 'xhigh'], default=medium)
  🟢 gpt-5.4  (visibility=list, reasoning=['low', 'medium', 'high', 'xhigh'], default=medium)
  🟢 gpt-5.2-codex  (visibility=list, reasoning=['low', 'medium', 'high', 'xhigh'], default=medium)
  🟢 gpt-5.1-codex-max  (visibility=list, reasoning=['low', 'medium', 'high', 'xhigh'], default=medium)
  ⚪ gpt-5.1-codex  (visibility=hide, reasoning=['low', 'medium', 'high'], default=medium)
  🟢 gpt-5.2  (visibility=list, reasoning=['low', 'medium', 'high', 'xhigh'], default=medium)
  ⚪ gpt-5.1  (visibility=hide, reasoning=['low', 'medium', 'high'], default=medium)
  ⚪ gpt-5-codex  (visibility=hide, reasoning=['low', 'medium', 'high'], default=medium)
  ⚪ gpt-5  (visibility=hide, reasoning=['minimal', 'low', 'medium', 'high'], default=medium)
  ⚪ gpt-oss-120b  (visibility=hide, reasoning=['low', 'medium', 'high'], default=medium)
  ⚪ gpt-oss-20b  (visibility=hide, reasoning=['low', 'm

## Cell 4 : Cursor — CLI 상태 확인


In [58]:
# cell 4 : Cursor — CLI 상태 확인
print("=" * 60)
print("Cursor CLI 상태")
print("=" * 60)

if not cli_results.get("cursor-agent", {}).get("installed"):
    print("❌ cursor-agent CLI 미설치 — 건너뜀")
else:
    rc, stdout, stderr = run_quiet(["cursor-agent", "--help"], timeout=30)
    combined = stdout + stderr
    print(f"cursor-agent --help (exit={rc}):")
    print(combined[:1000])
    print()
    print("Cursor 모델명은 Cursor Pro 구독 내 지원 모델을 사용합니다.")
    print("예: sonnet, gpt-5, composer 등 (cursor-agent --help에서 확인)")


Cursor CLI 상태
❌ cursor-agent CLI 미설치 — 건너뜀


## Cell 5 : Aider — 모델 목록 확인


In [59]:
# cell 5 : Aider — 모델 목록 확인
print("=" * 60)
print("Aider 모델 목록")
print("=" * 60)

if not cli_results.get("aider", {}).get("installed"):
    print("❌ aider CLI 미설치 — 건너뜀")
else:
    rc, stdout, stderr = run_quiet(["aider", "--list-models", ""], timeout=60)
    if rc == 0 and stdout:
        lines = [line.strip() for line in stdout.splitlines() if line.strip()]
        print(f"총 {len(lines)}개 모델 (상위 30개 표시):")
        for line in lines[:30]:
            print(f"  🟢 {line}")
        if len(lines) > 30:
            print(f"  ... +{len(lines) - 30}개")
    else:
        print("aider --list-models 실행 결과:")
        print((stdout or stderr)[:1000])
        print("\nAider 모델명 예시: openai/gpt-4o, anthropic/claude-3-7-sonnet-20250219")


Aider 모델 목록
❌ aider CLI 미설치 — 건너뜀


## Cell 6 : Antigravity — CLI 상태 확인


In [60]:
# cell 6 : Antigravity — CLI 상태 확인
print("=" * 60)
print("Antigravity CLI 상태")
print("=" * 60)

if not cli_results.get("antigravity", {}).get("installed"):
    print("❌ antigravity CLI 미설치 — 건너뜀")
else:
    rc, stdout, stderr = run_quiet(["antigravity", "chat", "--help"], timeout=30)
    combined = stdout + stderr
    has_mode = "--mode" in combined
    print(f"antigravity chat --help (exit={rc}):")
    print(combined[:1000])
    print()
    if has_mode:
        print("✅ --mode 옵션 확인됨")
    else:
        print("⚠️ --mode 옵션 미확인")
    print()
    print("주의: Antigravity CLI는 현재 --model 옵션을 노출하지 않습니다.")
    print("MODEL 값은 실험 기록 라벨이며, 실제 모델 선택은 UI에서 직접 수행합니다.")


Antigravity CLI 상태
antigravity chat --help (exit=0):
Antigravity 1.107.0

Usage: antigravity.exe chat [options] [prompt]

To read from stdin, append '-' (e.g. 'echo Hello World | antigravity.exe chat <prompt> -')

Options
  -m --mode <mode>        The mode to use for the chat session. Available
                          options: 'ask', 'edit', 'agent', or the identifier of
                          a custom mode. Defaults to 'agent'.
  -a --add-file <path>    Add files as context to the chat session.
  --maximize              Maximize the chat session view.
  -r --reuse-window       Force to use the last active window for the chat
                          session.
  -n --new-window         Force to open an empty window for the chat session.
  --profile <profileName> Opens the provided folder or workspace with the given
                          profile and associates the profile with the
                          workspace. If the profile does not exist, a new empty
                  

## Cell 7 : GitHub Copilot (gh) — CLI 상태 확인


In [61]:
# cell 7 : GitHub Copilot (gh) — CLI 상태 확인
print("=" * 60)
print("GitHub Copilot CLI 상태")
print("=" * 60)

if not cli_results.get("gh", {}).get("installed"):
    print("❌ gh CLI 미설치 — 건너뜀")
else:
    rc, stdout, stderr = run_quiet(["gh", "copilot", "--help"], timeout=30)
    if rc == 0:
        print("gh copilot --help:")
        print((stdout + stderr)[:1000])
    else:
        print(f"⚠️ gh copilot 실행 실패 (exit={rc})")
        print((stderr or stdout)[:500])
        print("\ngh auth login 후 gh extension install github/gh-copilot 필요할 수 있음")
    print()
    print("참고: Copilot CLI는 대화형 보조 위주이며, 코드 수정 비대화형 모드는 제한적입니다.")
    print("실험에는 manual.ps1 어댑터 사용을 권장합니다.")


GitHub Copilot CLI 상태
gh copilot --help:
Runs the GitHub Copilot CLI.

Executing the Copilot CLI through `gh` is currently in preview and subject to change.

If already installed, `gh` will execute the Copilot CLI found in your `PATH`.
If the Copilot CLI is not installed, it will be downloaded to C:\Users\Administrator\AppData\Local\GitHub CLI\copilot.

Use `--remove` to remove the downloaded Copilot CLI.

This command is only supported on Windows, Linux, and Darwin, on amd64/x64
or arm64 architectures.

To prevent `gh` from interpreting flags intended for Copilot,
use `--` before Copilot flags and args.

Learn more at https://gh.io/copilot-cli


USAGE
  gh copilot [flags] [args]

FLAGS
  --remove   Remove the downloaded Copilot CLI

INHERITED FLAGS
  --help   Show help for command

EXAMPLES
  # Download and run the Copilot CLI
  $ gh copilot
  
  # Run the Copilot CLI
  $ gh copilot -p "Summarize this week's commits" --allow-tool 'shell(git)'
  
  # Remove the Copilot CLI (

참고: Copil

## Cell 8 : Phase 1 권장 설정 예시 (Agent별)


In [62]:
# cell 8 : Phase 1 권장 설정 예시 (Agent별)
print("=" * 60)
print("Phase 1 Agent × Model 설정 예시")
print("=" * 60)

PHASE1_CONFIGS = [
    {
        "agent": "codex",
        "notebook": "phase1-codex-run.ipynb",
        "configs": [
            {'AGENT': 'codex', 'MODEL': 'gpt-5.4-mini',  'REASONING_EFFORT': 'low',     'CODEX_REASONING_FLAG': '--reasoning-effort'},
            {'AGENT': 'codex', 'MODEL': 'gpt-5.4-mini',  'REASONING_EFFORT': 'medium',  'CODEX_REASONING_FLAG': '--reasoning-effort'},
            {'AGENT': 'codex', 'MODEL': 'gpt-5.4-mini',  'REASONING_EFFORT': 'high',    'CODEX_REASONING_FLAG': '--reasoning-effort'},
            {'AGENT': 'codex', 'MODEL': 'o3-mini',       'REASONING_EFFORT': 'default', 'CODEX_REASONING_FLAG': '--reasoning-effort'},
        ],
    },
    {
        "agent": "cursor",
        "notebook": "phase1-cursor-run.ipynb",
        "configs": [
            {'AGENT': 'cursor', 'MODEL': 'sonnet',    'REASONING_EFFORT': 'default', 'CODEX_REASONING_FLAG': '--reasoning-effort'},
            {'AGENT': 'cursor', 'MODEL': 'gpt-5',     'REASONING_EFFORT': 'default', 'CODEX_REASONING_FLAG': '--reasoning-effort'},
            {'AGENT': 'cursor', 'MODEL': 'composer',  'REASONING_EFFORT': 'default', 'CODEX_REASONING_FLAG': '--reasoning-effort'},
        ],
    },
    {
        "agent": "aider",
        "notebook": "phase1-full-run.ipynb (or dedicated)",
        "configs": [
            {'AGENT': 'aider', 'MODEL': 'openai/gpt-4o',                          'REASONING_EFFORT': 'default', 'CODEX_REASONING_FLAG': '--reasoning-effort'},
            {'AGENT': 'aider', 'MODEL': 'openai/gpt-4o-mini',                     'REASONING_EFFORT': 'default', 'CODEX_REASONING_FLAG': '--reasoning-effort'},
            {'AGENT': 'aider', 'MODEL': 'anthropic/claude-3-7-sonnet-20250219',   'REASONING_EFFORT': 'default', 'CODEX_REASONING_FLAG': '--reasoning-effort'},
        ],
    },
    {
        "agent": "antigravity",
        "notebook": "phase1-antigravity-run.ipynb",
        "configs": [
            {'AGENT': 'antigravity', 'MODEL': 'gemini-3-flash',       'REASONING_EFFORT': 'default', 'CODEX_REASONING_FLAG': '--reasoning-effort'},
            {'AGENT': 'antigravity', 'MODEL': 'gemini-3.1-pro-preview', 'REASONING_EFFORT': 'default', 'CODEX_REASONING_FLAG': '--reasoning-effort'},
            {'AGENT': 'antigravity', 'MODEL': 'claude-opus-4.6',      'REASONING_EFFORT': 'default', 'CODEX_REASONING_FLAG': '--reasoning-effort'},
        ],
    },
    {
        "agent": "manual (copilot GUI 등)",
        "notebook": "phase1-full-run.ipynb (or dedicated)",
        "configs": [
            {'AGENT': 'manual', 'MODEL': 'copilot-gui-claude', 'REASONING_EFFORT': 'default', 'CODEX_REASONING_FLAG': '--reasoning-effort'},
        ],
    },
]

for group in PHASE1_CONFIGS:
    agent = group['agent']
    nb = group['notebook']
    # CLI 설치 상태
    cli_key = {'codex': 'codex', 'cursor': 'cursor-agent', 'aider': 'aider',
               'antigravity': 'antigravity', 'manual (copilot GUI 등)': 'gh'}.get(agent, '')
    installed = cli_results.get(cli_key, {}).get('installed', False)
    icon = '✅' if installed else '❌'
    print(f"\n{'─' * 60}")
    print(f"{icon} Agent: {agent}  |  Notebook: {nb}")
    print(f"{'─' * 60}")
    for i, cfg in enumerate(group['configs'], 1):
        print(f"  [{i}] AGENT = \"{cfg['AGENT']}\"")
        print(f"      MODEL = \"{cfg['MODEL']}\"")
        print(f"      REASONING_EFFORT = \"{cfg['REASONING_EFFORT']}\"")
        print(f"      CODEX_REASONING_FLAG = \"{cfg['CODEX_REASONING_FLAG']}\"")
        print()


Phase 1 Agent × Model 설정 예시

────────────────────────────────────────────────────────────
✅ Agent: codex  |  Notebook: phase1-codex-run.ipynb
────────────────────────────────────────────────────────────
  [1] AGENT = "codex"
      MODEL = "gpt-5.4-mini"
      REASONING_EFFORT = "low"
      CODEX_REASONING_FLAG = "--reasoning-effort"

  [2] AGENT = "codex"
      MODEL = "gpt-5.4-mini"
      REASONING_EFFORT = "medium"
      CODEX_REASONING_FLAG = "--reasoning-effort"

  [3] AGENT = "codex"
      MODEL = "gpt-5.4-mini"
      REASONING_EFFORT = "high"
      CODEX_REASONING_FLAG = "--reasoning-effort"

  [4] AGENT = "codex"
      MODEL = "o3-mini"
      REASONING_EFFORT = "default"
      CODEX_REASONING_FLAG = "--reasoning-effort"


────────────────────────────────────────────────────────────
❌ Agent: cursor  |  Notebook: phase1-cursor-run.ipynb
────────────────────────────────────────────────────────────
  [1] AGENT = "cursor"
      MODEL = "sonnet"
      REASONING_EFFORT = "default"
    

## Cell 9 : 종합 요약


In [63]:
# cell 9 : 종합 요약
print("=" * 60)
print("Phase 0 종합 요약")
print(f"점검 일시: {CHECK_DATE}")
print("=" * 60)

summary_rows = []
for cli_name, r in cli_results.items():
    summary_rows.append({
        "CLI": cli_name,
        "설치": "✅" if r["installed"] else "❌",
        "버전": (r["version"] or "-")[:50],
        "경로": (r["path"] or "-")[:60],
    })

try:
    import pandas as pd
    df = pd.DataFrame(summary_rows)
    display(df)
except ImportError:
    for row in summary_rows:
        print(f"  {row['설치']} {row['CLI']:20s} {row['버전']}")

print()
if codex_models:
    visible = [m['slug'] for m in codex_models if m['visibility'] == 'list']
    print(f"Codex visible 모델: {visible}")
else:
    print("Codex 모델 정보 없음 (미설치 또는 조회 실패)")

print()
print("다음 단계:")
print("  1. 위 결과에서 사용할 agent × model 조합을 결정")
print("  2. 해당 Phase 1 노트북의 00 Configuration 셀에 값을 설정")
print("  3. Phase 1 노트북을 위에서부터 순서대로 실행")


Phase 0 종합 요약
점검 일시: 2026-04-25 09:55:13


,CLI,설치,버전,경로
0,codex,✅,codex-cli 0.122.0,C:\Users\Administrator\AppData\Roaming\npm\cod...
1,cursor-agent,❌,-,-
2,aider,❌,-,-
3,antigravity,✅,1.107.0\n15487b3041e65228cae24980a3f796c905ef5...,C:\Program Files\Antigravity\bin\antigravity.CMD
4,gh,✅,gh version 2.86.0 (2026-01-21)\nhttps://github...,C:\Program Files\GitHub CLI\gh.EXE



Codex visible 모델: ['gpt-5.3-codex', 'gpt-5.4', 'gpt-5.2-codex', 'gpt-5.1-codex-max', 'gpt-5.2', 'gpt-5.1-codex-mini', 'gpt-5.4-mini']

다음 단계:
  1. 위 결과에서 사용할 agent × model 조합을 결정
  2. 해당 Phase 1 노트북의 00 Configuration 셀에 값을 설정
  3. Phase 1 노트북을 위에서부터 순서대로 실행
